# 03 — Novelty Clustering: HDBSCAN pada Flow Unknown (Paper 3, Tahap T3b)

**Dijalankan di SageMaker.** Menjawab: *kumpulan flow `unknown` (hasil open-set nb 02)
membentuk berapa kelas serangan baru?* Idealnya bila diinjeksi **M jenis baru**, muncul
**≈ M cluster** padat & terpisah.

**Alur:**
1. Muat artefak nb 01 (`deploy_meta_mc_<DS>.json`: scaler + mu + inv_cov) + model.
2. Bentuk himpunan **unknown**: flow kelas **held-out** yang **lolos gerbang open-set**
   (Mahalanobis min-dist > τ, τ = p99 jarak known dari nb 02). Ini meniru kondisi nyata:
   yang di-cluster hanya yang benar-benar ditandai unknown, bukan semua held-out.
3. **HDBSCAN** di ruang 9-fitur ter-scale → temukan cluster padat; noise (label -1) dibuang.
4. Evaluasi: jumlah cluster vs M (jumlah jenis held-out), **homogeneity** & **ARI**
   terhadap ground-truth `attack_cat` held-out (label lab = oracle).

**Output** (→ S3 `evolusion/novelty/`): `novelty_results.json`, `novelty_<DS>.png`
(proyeksi 2D cluster), `cluster_profile_<DS>.csv` (profil statistik tiap cluster —
bekal pelabelan LLM di nb 05).

> Catatan kejujuran (documentation.md §5): cluster ≠ selalu 1 jenis serangan (bisa
> menyatu/pecah). Dilaporkan apa adanya via homogeneity/ARI. Semua angka dari eksekusi nyata.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('xgboost','scikit-learn','scipy','pandas','numpy','matplotlib','boto3','hdbscan') if u.find_spec(m.replace('scikit-learn','sklearn')) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import homogeneity_score, adjusted_rand_score
from sklearn.decomposition import PCA
import xgboost as xgb
try:
    import hdbscan; HAVE_HDBSCAN=True
except Exception:
    from sklearn.cluster import DBSCAN; HAVE_HDBSCAN=False; print('hdbscan tak ada -> fallback DBSCAN')
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
IN01='known_base_out'; OUTDIR='novelty_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; MIN_CLASS=200
HELDOUT={'CIC':['Botnet','Infiltration'],'UNSW':['Worms','Shellcode','Backdoor']}  # SAMA dgn nb 01/02
MIN_CLUSTER_SIZE=50  # HDBSCAN: cluster minimal (padat); sesuaikan bila perlu
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED,
         'heldout':HELDOUT,'min_cluster_size':MIN_CLUSTER_SIZE,'used_hdbscan':HAVE_HDBSCAN}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
def fetch01(fname):
    lp=os.path.join(IN01,fname)
    if os.path.exists(lp): return lp
    try:
        import boto3; os.makedirs(IN01,exist_ok=True)
        boto3.client('s3',region_name=REGION).download_file(S3_BUCKET,f'{S3_PREFIX}/known_base/{fname}',lp)
        print('   diunduh dari S3:',fname); return lp
    except Exception as e:
        print('   GAGAL ambil',fname,'->',e); return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Loader data + mapping label (identik nb 01/02)

In [ ]:
def map_cic(lbl):
    s=str(lbl).strip().lower()
    if s in ('benign','normal'): return 'Benign'
    if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
    if s.startswith('dos'): return 'DoS'
    if 'bruteforce' in s or 'brute force' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
    if s=='bot' or 'botnet' in s: return 'Botnet'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'Web'
    return 'Other'
def map_uns(lbl):
    s=str(lbl).strip().lower()
    if s in ('normal','benign',''): return 'Benign'
    return {'dos':'DoS','exploits':'Exploits','fuzzers':'Fuzzers','generic':'Generic',
            'reconnaissance':'Recon','backdoor':'Backdoor','backdoors':'Backdoor',
            'shellcode':'Shellcode','worms':'Worms','analysis':'Analysis'}.get(s,'Other')

def load_cic():
    p=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
    if not p: print('CIC csv tak ada'); return None
    c=pd.read_csv(p, low_memory=False); c.columns=c.columns.str.strip()
    cm={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
        'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    if not all(v in c.columns for v in cm.values()): print('CIC kolom kurang'); return None
    d=pd.DataFrame({k:pd.to_numeric(c[cm[k]],errors='coerce') for k in CANON})
    d['cat']=c[LAB].map(map_cic)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

def _pick_unsw():
    cands=[]
    for pat in ['../data/UNSW_NB15_*set.csv','../../unswnb-15/data/UNSW_NB15_*set.csv']:
        cands+=sorted(glob.glob(pat))
    cands=list(dict.fromkeys(cands))
    if not cands: return None
    best,best_n=None,-1
    for p in cands:
        try: n=sum(1 for _ in open(p,'r',errors='ignore'))-1
        except Exception: n=-1
        if n>best_n: best,best_n=p,n
    print(f'    UNSW dipakai: {os.path.basename(best)} (~{best_n} record)'); return best

def load_uns():
    p=_pick_unsw()
    if not p: print('UNSW csv tak ada'); return None
    u2=pd.read_csv(p); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','attack_cat']
    if not all(x in u2.columns for x in need): print('UNSW kolom kurang'); return None
    d=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                    'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                    'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
    d['cat']=u2['attack_cat'].fillna('Normal').map(map_uns)
    d=d.replace([np.inf,-np.inf],np.nan).dropna(); d=d[d['cat']!='Other']
    return d

cic=load_cic(); uns=load_uns()
print('=== SEL 2 (loader) SELESAI ===')

## 3. Bentuk himpunan unknown (held-out yang lolos gerbang open-set) + cluster

In [ ]:
def load_art(ds):
    mp=fetch01(f'deploy_meta_mc_{ds}.json')
    if not mp: return None
    meta=json.load(open(mp)); labels=meta['labels']
    return dict(meta=meta,mean=np.asarray(meta['scaler_mean'],float),scale=np.asarray(meta['scaler_scale'],float),
                labels=labels,
                mus=np.array([meta['class_stats'][c]['mu'] for c in labels],float),
                icovs=np.array([meta['class_stats'][c]['inv_cov'] for c in labels],float))

def scale_X(art,X): return np.nan_to_num((X-art['mean'])/art['scale'],nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)

def maha_min(art,Xs):
    dmin=np.full(Xs.shape[0],np.inf)
    for k in range(art['mus'].shape[0]):
        diff=Xs-art['mus'][k]; d2=np.clip(np.einsum('ni,ij,nj->n',diff,art['icovs'][k],diff),0,None)
        dmin=np.minimum(dmin,np.sqrt(d2))
    return dmin

def merge_rare(df,min_n=MIN_CLASS):
    vc=df['cat'].value_counts(); rare=[c for c,n in vc.items() if n<min_n and c!='Benign']
    if rare: df=df.copy(); df['cat']=df['cat'].where(~df['cat'].isin(rare),'Other-rare')
    return df

def tau_from_known(art,df,ds):
    """Rekonstruksi τ = p99 jarak known (test split identik nb 01/02)."""
    heldout=HELDOUT.get(ds,[]); dk=merge_rare(df[~df['cat'].isin(heldout)].copy())
    dk=dk[dk['cat'].isin(art['labels'])]
    le=LabelEncoder().fit(art['labels']); y=le.transform(dk['cat'].values)
    _,Xte,_,_=train_test_split(dk[CANON].values,y,test_size=0.3,random_state=SEED,stratify=y)
    return float(np.percentile(maha_min(art,scale_X(art,Xte)),99))

def cluster(Xs):
    if HAVE_HDBSCAN:
        cl=hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,min_samples=10)
        return cl.fit_predict(Xs)
    return DBSCAN(eps=1.5,min_samples=MIN_CLUSTER_SIZE).fit_predict(Xs)
print('=== SEL 3 (fungsi unknown + cluster) SELESAI ===')

## 4. Jalankan per dataset: filter unknown → HDBSCAN → evaluasi vs ground-truth

In [ ]:
def run_novelty(ds,df):
    if df is None: print(f'[{ds}] data kosong'); return None
    art=load_art(ds)
    if art is None: print(f'[{ds}] artefak 01 tak ada; jalankan nb 01 dulu'); return None
    heldout=HELDOUT.get(ds,[]); M=len(heldout)
    dh=df[df['cat'].isin(heldout)].copy()
    if len(dh)==0: print(f'[{ds}] tak ada flow held-out'); return None
    tau=tau_from_known(art,df,ds)
    Xh_s=scale_X(art,dh[CANON].values); dist=maha_min(art,Xh_s)
    mask=dist>tau                         # hanya yg LOLOS gerbang open-set (ditandai unknown)
    Xu=Xh_s[mask]; gt=dh['cat'].values[mask]
    print(f"[{ds}] held-out={heldout} M={M} | n_heldout={len(dh)} lolos-unknown={mask.sum()} "
          f"(tau={tau:.2f})")
    if len(Xu)<MIN_CLUSTER_SIZE: print('   unknown terlalu sedikit utk cluster'); return None
    lab=cluster(Xu)
    valid=lab[lab>=0]; n_clusters=len(set(valid.tolist())); n_noise=int((lab==-1).sum())
    hom=float(homogeneity_score(gt,lab)); ari=float(adjusted_rand_score(gt,lab))
    # komposisi ground-truth per cluster (untuk lihat cluster murni/campur)
    comp={}
    for cl_id in sorted(set(valid.tolist())):
        vals,cnts=np.unique(gt[lab==cl_id],return_counts=True)
        comp[int(cl_id)]={str(v):int(n) for v,n in zip(vals,cnts)}
    res=dict(dataset=ds,heldout=heldout,M_expected=M,n_heldout=int(len(dh)),n_unknown=int(mask.sum()),
             tau=round(tau,4),n_clusters=n_clusters,n_noise=n_noise,
             homogeneity=round(hom,4),ari=round(ari,4),cluster_composition=comp)
    print(f"   -> cluster={n_clusters} (expected~{M}) noise={n_noise} | homogeneity={hom:.3f} ARI={ari:.3f}")
    # profil statistik tiap cluster (bekal LLM auto-name nb 05) di ruang ASLI (un-scale)
    Xu_orig=Xu*art['scale']+art['mean']
    prof=pd.DataFrame(Xu_orig,columns=CANON); prof['cluster']=lab; prof=prof[prof['cluster']>=0]
    prof_stat=prof.groupby('cluster').median().round(3); prof_stat['n']=prof.groupby('cluster').size()
    prof_stat.to_csv(os.path.join(OUTDIR,f'cluster_profile_{ds}.csv'))
    print(f'   profil cluster -> cluster_profile_{ds}.csv')
    # plot proyeksi PCA 2D
    try:
        p2=PCA(n_components=2,random_state=SEED).fit_transform(Xu)
        fig,ax=plt.subplots(figsize=(6,5))
        sc=ax.scatter(p2[:,0],p2[:,1],c=lab,cmap='tab10',s=6,alpha=0.6)
        ax.set_title(f'{ds}: cluster unknown (PCA 2D)\ncluster={n_clusters} exp~{M} hom={hom:.2f} ARI={ari:.2f}')
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); fig.colorbar(sc,ax=ax,label='cluster (-1=noise)')
        plt.tight_layout(); savefig(f'novelty_{ds}.png')
    except Exception as e: print('   plot gagal:',e)
    return res

RESULTS['novelty']={}
for ds,df in [('CIC',cic),('UNSW',uns)]:
    r=run_novelty(ds,df)
    if r: RESULTS['novelty'][ds]=r
print('=== SEL 4 (novelty clustering) SELESAI ===')

## 5. Ringkas + simpan + UPLOAD S3

In [ ]:
rows=[]
for ds,r in RESULTS.get('novelty',{}).items():
    rows.append({'dataset':ds,'held_out':','.join(r['heldout']),'M_expected':r['M_expected'],
                 'n_unknown':r['n_unknown'],'n_clusters':r['n_clusters'],'noise':r['n_noise'],
                 'homogeneity':r['homogeneity'],'ARI':r['ari']})
summ=pd.DataFrame(rows)
import IPython.display as ipd; print('Ringkasan novelty clustering:'); ipd.display(summ)
summ.to_csv(os.path.join(OUTDIR,'novelty_summary.csv'),index=False)
jp=os.path.join(OUTDIR,'novelty_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/novelty/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/novelty/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan + upload) SELESAI ===')
print('SELESAI T3b. n_clusters~M & homogeneity tinggi -> cluster = kandidat kelas baru bersih. '
      'cluster_profile_<DS>.csv jadi input pelabelan (oracle + LLM) di nb 05.')